# 01 · arXiv — graph + vector in one store

A property graph of papers, authors and categories, with pgvector HNSW over the
same Paper entities. This notebook tours **query, read and mutation over one
store** — Cypher analytics, vector search, graph expansion and GraphRAG, then
the `get`/`get_triplets` read API and a full upsert → get → delete lifecycle.

> Run `prepare.py` first to build the `arxiv` graph (this notebook reads it).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

# open the existing graph (built by prepare.py); vector_dimension lets vector_query use HNSW
store = agens.make_pg_store("arxiv", vector_dimension=EMBED_DIM, create=False)
embed_model, llm = get_embed_model(), get_llm()

## The graph

Every node lives on one `"__Node__"` label; the entity type is an indexed
`__type__` scalar. Counting by type uses that index + `count(*)`:

In [2]:
import pandas as pd
rows = []
for t in ("Paper", "Author", "Category"):
    n = store.structured_query(
        "MATCH (n:\"__Node__\") WHERE n.__type__ = %(t)s RETURN count(*) AS c",
        {"t": __import__('psycopg').types.json.Jsonb(t)})[0]["c"]
    rows.append({"type": t, "count": n})
edges = store.structured_query('MATCH (:\"__Node__\")-[r]->(:\"__Node__\") RETURN count(*) AS c')[0]["c"]
print(f"relationships: {edges:,}")
pd.DataFrame(rows)

relationships: 224,746


,type,count
0,Paper,50000
1,Author,82616
2,Category,147


## (a) Analytics — plain Cypher

Aggregations walk the **edges** (cheap — the edge implies the endpoint type) and
use `count(*)` (not `count(p)`, which would materialize each paper's embedding).

In [3]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__")-[:"AUTHORED_BY"]->(a:"__Node__")
    RETURN a.name AS author, count(*) AS papers ORDER BY papers DESC LIMIT 10'''))

,author,papers
0,Damien Chablat,84
1,B. Aubert,66
2,H. Vincent Poor,63
3,The BABAR Collaboration,61
4,Philippe Wenger,60
5,CDF Collaboration,42
6,F. Combes,34
7,D0 Collaboration,32
8,S. Das Sarma,32
9,N. Gehrels,31


In [4]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__")-[:"IN_CATEGORY"]->(c:"__Node__")
    RETURN c.name AS category, count(*) AS papers ORDER BY papers DESC LIMIT 10'''))

,category,papers
0,astro-ph,10250
1,hep-ph,4806
2,hep-th,4552
3,quant-ph,3155
4,gr-qc,2714
5,cond-mat.mtrl-sci,2214
6,cond-mat.stat-mech,2188
7,math.MP,2167
8,math-ph,2167
9,cond-mat.str-el,2013


In [5]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__") WHERE p.year IS NOT NULL
    RETURN p.year AS year, count(*) AS papers ORDER BY year DESC LIMIT 10'''))

,year,papers
0,2023,29
1,2022,62
2,2021,73
3,2020,107
4,2019,532
5,2018,110
6,2017,312
7,2016,579
8,2015,1290
9,2014,749


## (b) Vector search — HNSW over Paper abstracts

`vector_query` embeds the question and runs an HNSW nearest-neighbour search,
returning entities + cosine scores.

In [6]:
from llama_index.core.vector_stores.types import VectorStoreQuery
question = "graph neural networks for molecular property prediction"
qv = embed_model.get_query_embedding(question)
nodes, scores = store.vector_query(VectorStoreQuery(query_embedding=qv, similarity_top_k=5))
for n, s in zip(nodes, scores):
    print(f"{s:.3f}  {(n.properties or {}).get('title', n.name)[:80]}")

0.490  Virtual screening of GPCRs: an in silico chemogenomics approach
0.485  Kernel methods for in silico chemogenomics
0.470  A System for Predicting Subcellular Localization of Yeast Genome Using Neural Ne
0.460  Validating module network learning algorithms using simulated data
0.458  Improved Neural Modeling of Real-World Systems Using Genetic Algorithm Based Var


## (c) Graph expansion — `get_rel_map`

Expand the vector hits through the graph (their authors and categories).

In [7]:
for src, rel, tgt in store.get_rel_map(nodes[:3], depth=1, limit=15):
    sn = (src.properties or {}).get('title', src.name)
    print(f"({str(sn)[:38]}) -[{rel.label}]-> ({tgt.name})")

(A System for Predicting Subcellular Lo) -[AUTHORED_BY]-> (Sabu M. Thampi)
(A System for Predicting Subcellular Lo) -[AUTHORED_BY]-> (K. Chandra Sekaran)
(A System for Predicting Subcellular Lo) -[IN_CATEGORY]-> (cs.NE)
(A System for Predicting Subcellular Lo) -[IN_CATEGORY]-> (cs.AI)
(Kernel methods for in silico chemogeno) -[AUTHORED_BY]-> (Jean-Philippe Vert)
(Kernel methods for in silico chemogeno) -[AUTHORED_BY]-> (Laurent Jacob)
(Kernel methods for in silico chemogeno) -[IN_CATEGORY]-> (q-bio.QM)
(Virtual screening of GPCRs: an in sili) -[AUTHORED_BY]-> (Jean-Philippe Vert)
(Virtual screening of GPCRs: an in sili) -[AUTHORED_BY]-> (Laurent Jacob)
(Virtual screening of GPCRs: an in sili) -[AUTHORED_BY]-> (Brice Hoffmann)
(Virtual screening of GPCRs: an in sili) -[AUTHORED_BY]-> (Véronique Stoven)
(Virtual screening of GPCRs: an in sili) -[IN_CATEGORY]-> (q-bio.QM)


## (d) GraphRAG

Attach a `PropertyGraphIndex` to the populated store (`kg_extractors=[]` so it
never re-extracts) and answer with a grounded LLM response.

In [8]:
from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import VectorContextRetriever
index = PropertyGraphIndex.from_existing(store, embed_model=embed_model, llm=llm,
                                         kg_extractors=[], use_async=False)
qe = index.as_query_engine(sub_retrievers=[
    VectorContextRetriever(graph_store=store, embed_model=embed_model,
                           similarity_top_k=5, path_depth=1, include_text=True)], llm=llm)
print(qe.query(question))

Graph neural networks are increasingly being utilized in the field of molecular property prediction, leveraging their ability to model complex relationships and interactions within molecular structures. These networks can effectively capture the graph-like nature of molecules, allowing for improved predictions of various properties based on the molecular graph representation.


## (e) Read by id & triplets — `get` / `get_triplets`

Fetch specific nodes by id (or property) and the triplets around them — the
read side of the property-graph API, reusing the vector hits from (b) as seeds.

In [9]:
ids = [n.name for n in nodes[:3]]
for n in store.get(ids=ids):                       # fetch specific Paper nodes by id
    print(f"[{n.name}] {(n.properties or {}).get('title', n.name)[:70]}")
print("\ntriplets leaving the top paper:")
for s, r, t in store.get_triplets(entity_names=ids[:1]):
    print(f"  ({s.name}) -[{r.label}]-> ({t.name})")

[0710.2227] A System for Predicting Subcellular Localization of Yeast Genome Using
[0709.3931] Kernel methods for in silico chemogenomics
[0801.4301] Virtual screening of GPCRs: an in silico chemogenomics approach

triplets leaving the top paper:


  (0801.4301) -[AUTHORED_BY]-> (Jean-Philippe Vert)
  (0801.4301) -[AUTHORED_BY]-> (Laurent Jacob)
  (0801.4301) -[AUTHORED_BY]-> (Brice Hoffmann)
  (0801.4301) -[AUTHORED_BY]-> (Véronique Stoven)
  (0801.4301) -[IN_CATEGORY]-> (q-bio.QM)


## (f) Mutation lifecycle — `upsert` / `get` / `delete`

A throwaway `crud_demo` graph (on its own `from_conf` engine) so the populated
`arxiv` graph is never mutated: upsert a few nodes, read them back by property
and via triplets, delete by id and by name, then clear.

In [10]:
from llama_index.core.graph_stores.types import EntityNode, Relation
from llama_index_agensgraph.engine import AgensEngine
from llama_index_agensgraph.graph_stores.agensgraph import AgensPropertyGraphStore

scratch_engine = AgensEngine.from_conf(config.conf())   # from_conf: build a pool from a conf dict
scratch = AgensPropertyGraphStore("crud_demo", conf=config.conf(), vector_dimension=EMBED_DIM,
                                  create=True, refresh_schema=False, engine=scratch_engine)
scratch.structured_query('MATCH (n:"__Node__") DETACH DELETE n')  # reset

scratch.upsert_nodes([
    EntityNode(name="demo:p1", label="Paper", properties={"title": "Graphs for X", "year": 2024}),
    EntityNode(name="demo:p2", label="Paper", properties={"title": "Graphs for Y", "year": 2025}),
    EntityNode(name="demo:alice", label="Author")])
scratch.upsert_relations([
    Relation(label="AUTHORED_BY", source_id="demo:p1", target_id="demo:alice"),
    Relation(label="AUTHORED_BY", source_id="demo:p2", target_id="demo:alice")])

print("get(properties={'year': 2025}):", [n.name for n in scratch.get(properties={"year": 2025})])
print("triplets:", [(s.name, r.label, t.name)
                    for s, r, t in scratch.get_triplets(entity_names=["demo:p1", "demo:p2"])])
scratch.delete(ids=["demo:p1"]); scratch.delete(entity_names=["demo:alice"])
print("remaining:", [n.name for n in scratch.get(ids=["demo:p1", "demo:p2", "demo:alice"])])
scratch.structured_query('MATCH (n:"__Node__") DETACH DELETE n')  # leave it empty
scratch_engine.close()

get(properties={'year': 2025}): ['demo:p2']
triplets: [('demo:p1', 'AUTHORED_BY', 'demo:alice'), ('demo:p2', 'AUTHORED_BY', 'demo:alice')]
remaining: ['demo:p2']


## How it was built

`prepare.py` streams arXiv from Hugging Face and ingests it directly (no LLM
extraction), then embeds the papers in parallel:

```python
store.upsert_nodes([EntityNode(name=paper_id, label="Paper",
                               properties={"title": t, "abstract": a, "year": y}),
                    EntityNode(name=author, label="Author")])
store.upsert_relations([Relation(label="AUTHORED_BY", source_id=paper_id, target_id=author)])
await store.aupsert_nodes([EntityNode(name=paper_id, label="Paper", embedding=vec)])
```

One AgensGraph graph now serves analytics, vector search, graph expansion and
GraphRAG — no separate graph DB and vector DB.

In [11]:
agens.close()